# Exercise Sheet 5 — Data as a Signal versus Noise
**Course:** Big Data Science in Finance (WI3436TU)
**Data:** simulated random matrices; Sheet 3 market data (`../Lab_3/E3/market_data_2007_2025.csv`)
for SPY, QQQ, TLT, HYG over 2018–2019

This notebook works through Problems 1-3 of Exercise Sheet 5. Code cells compute the required
quantities and produce the plots; markdown cells contain the hand calculations and written
discussion.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

SEED = 42
rng = np.random.default_rng(SEED)

MARKET_PATH = Path("../Lab_3/E3/market_data_2007_2025.csv")

### Notation

Let the following to be a return matrix: $\mathbf R \in \mathbb R^{n \times m}$. Center each row
to obtain $\mathbf X$, and use divisor $m$: $\widehat{\mathbf C} = \mathbf X\mathbf X^\top/m$. Order
eigenvalues $\lambda_1 \ge \dots \ge \lambda_n \ge 0$. For iid noise of variance $\sigma^2$, write
$\gamma = n/m$, $a = \sigma^2(1 - \sqrt\gamma)^2$, $b = \sigma^2(1 + \sqrt\gamma)^2$. The
Marchenko–Pastur (MP) continuous density is given by
$$f_{\gamma,\sigma^2}(x) = \frac{\sqrt{(b - x)(x - a)}}{2\pi\gamma\sigma^2 x}\mathbf 1_{\{a < x < b\}},$$
with an atom of mass $(1 - 1/\gamma)_+$ at zero.

For a real symmetric matrix, `np.linalg.eigvalsh(S)` returns eigenvalues in ascending order. Use
`np.linalg.eigh(S)` when eigenvectors are needed.

In [ ]:
def mp_edges(gamma, sigma2=1.0):
    """Marchenko-Pastur support edges a and b."""
    a = sigma2 * (1 - np.sqrt(gamma))**2
    b = sigma2 * (1 + np.sqrt(gamma))**2
    return a, b


def mp_density(x, gamma, sigma2=1.0):
    """Continuous part of the MP density (zero outside (a, b))."""
    a, b = mp_edges(gamma, sigma2)
    x = np.asarray(x, dtype=float)
    f = np.zeros_like(x)
    inside = (x > a) & (x < b)
    f[inside] = np.sqrt((b - x[inside]) * (x[inside] - a)) / (2 * np.pi * gamma * sigma2 * x[inside])
    return f


def mp_zero_mass(gamma):
    """Mass of the MP atom at zero, (1 - 1/gamma)_+."""
    return max(1 - 1 / gamma, 0.0)

## Problem 1

### a)

Classify the following as a Wigner matrix or not, with a reason.

i) A $3 \times 3$ matrix has zero diagonal and three independent signs
$\xi_{12}, \xi_{13}, \xi_{23} \in \{-1, +1\}$, each sign equally likely, mirrored below the diagonal.

ii) A $3 \times 3$ matrix has zero diagonal and the same random sign $\xi \in \{-1, +1\}$ in every
off-diagonal position.

**Answer.**

### b)

One realization of construction i) is
$$\mathbf W = \begin{pmatrix} 0 & 1 & 1 \\ 1 & 0 & 1 \\ 1 & 1 & 0 \end{pmatrix}.$$
Find the eigenvalues of $\mathbf W/\sqrt 3$ by hand and write its empirical spectral measure.

**Hand calculation.**

In [ ]:
# numerical check of the hand calculation
W = np.array([[0.0, 1.0, 1.0],
              [1.0, 0.0, 1.0],
              [1.0, 1.0, 0.0]])

print("eigenvalues of W / sqrt(3):", np.linalg.eigvalsh(W / np.sqrt(3)))
print("2/sqrt(3) =", 2 / np.sqrt(3), "  -1/sqrt(3) =", -1 / np.sqrt(3))

### c)

Let $\mathbf X \in \mathbb R^{n \times m}$ and $\mathbf S = \mathbf X\mathbf X^\top/m$. For
$n = 400$ and $m = 200$, how many zero eigenvalues must $\mathbf S$ have if $\mathbf X$ has rank 200?

**Answer.**

### d)

Calculate the MP support $a = \sigma^2(1 - \sqrt\gamma)^2$, and $b = \sigma^2(1 + \sqrt\gamma)^2$
and the mass at zero for $(\gamma, \sigma^2) = (2, 1)$.

In [ ]:
gamma, sigma2 = 2, 1
a, b = mp_edges(gamma, sigma2)

print(f"a = (1 - sqrt(2))^2 = {a:.4f}")
print(f"b = (1 + sqrt(2))^2 = {b:.4f}")
print(f"mass at zero = (1 - 1/2)_+ = {mp_zero_mass(gamma):.4f}")

## Problem 2

### a)

Generate a zero-diagonal Wigner matrix with $n = 500$ and independent $N(0, 1)$ upper entries.
Mirror the upper triangle, divide by $\sqrt n$, and plot the two eigenvalue histograms beside the
semicircle density $\rho(x) = \frac{1}{2\pi}\sqrt{4 - x^2}\,\mathbf 1_{\{|x| \le 2\}}$.

In [ ]:
n = 500
G = rng.normal(size=(n, n))
upper = np.triu(G, k=1)          # strictly upper triangle -> zero diagonal
W = upper + upper.T              # mirror the upper triangle
W_scaled = W / np.sqrt(n)

eig_W = np.linalg.eigvalsh(W)
eig_W_scaled = np.linalg.eigvalsh(W_scaled)

x = np.linspace(-2, 2, 400)
semicircle = np.sqrt(4 - x**2) / (2 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(eig_W, bins=50, density=True, alpha=0.7)
axes[0].plot(x, semicircle, color="red", lw=2, label="semicircle")
axes[0].set_title("eigenvalues of W (not scaled)")
axes[0].legend()

axes[1].hist(eig_W_scaled, bins=50, density=True, alpha=0.7)
axes[1].plot(x, semicircle, color="red", lw=2, label="semicircle")
axes[1].set_title("eigenvalues of W / sqrt(n)")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("eigenvalue")
    ax.set_ylabel("density")
fig.tight_layout()
plt.show()

print(f"range of eigenvalues of W/sqrt(n): [{eig_W_scaled.min():.3f}, {eig_W_scaled.max():.3f}]")

**Discussion.**

### b)

Generate an iid Gaussian matrix $\mathbf X$ with known zero mean and variance one for
$(n, m) = (400, 200)$. Estimate $\mathbf S = \frac{1}{m}\mathbf X\mathbf X^\top$ and plot its
eigenvalues together with the correctly scaled MP density. Mark the edges $a$ and $b$. Analyze
the number of zero eigenvalues and explain why the semicircle law is not appropriate for
analyzing these matrices.

In [ ]:
n, m = 400, 200
gamma = n / m
a, b = mp_edges(gamma)

X = rng.normal(size=(n, m))     # known zero mean: no centering needed
S = X @ X.T / m
eig_S = np.linalg.eigvalsh(S)

n_zero = np.sum(eig_S < 1e-10)
print(f"gamma = {gamma}, a = {a:.4f}, b = {b:.4f}")
print(f"number of (numerically) zero eigenvalues: {n_zero} of {n}  "
      f"(fraction {n_zero / n:.2f}, MP atom mass = {mp_zero_mass(gamma):.2f})")

# density histogram over ALL n eigenvalues: the nonzero part has mass 1/gamma,
# which matches the continuous MP density (it also integrates to 1/gamma)
x = np.linspace(1e-6, 1.1 * b, 600)
f = mp_density(x, gamma)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(eig_S, bins=60, range=(0, 1.1 * b), density=True, alpha=0.7, label="eigenvalues of S")
ax.plot(x, f, color="red", lw=2, label="MP density (continuous part)")
ax.axvline(a, color="black", ls="--", label=f"a = {a:.3f}")
ax.axvline(b, color="black", ls=":", label=f"b = {b:.3f}")
ax.set_ylim(0, 1.5 * f.max())    # the bar at 0 (atom of mass 1/2) is cut off
ax.set_xlabel("eigenvalue")
ax.set_ylabel("density")
ax.set_title("Eigenvalues of S = XX^T/m, (n, m) = (400, 200)")
ax.legend()
plt.show()

**Discussion.**

### c)

The code below is intended for variables in rows, dates in columns, and iid Gaussian noise of
variance 4. Identify and correct every line that is incompatible with this interpretation. State
the dimension of the corrected covariance matrix and the correct value of $\gamma$.

```python
n, m = 120, 240
R = 2 * rng.normal(size=(n, m))
X = R - R.mean(axis=0, keepdims=True)
S = X.T @ X / n
eig = np.linalg.eigvals(S) / np.sqrt(n)
gamma = m / n
a, b = (1 - np.sqrt(gamma))**2, (1 + np.sqrt(gamma))**2
```

**Discussion.**

In [ ]:
# corrected version
n, m = 120, 240
sigma2 = 4
R = 2 * rng.normal(size=(n, m))                  # std 2 -> variance 4
X = R - R.mean(axis=1, keepdims=True)            # center each variable (row) over the dates
S = X @ X.T / m                                  # n x n covariance with divisor m
eig = np.linalg.eigvalsh(S)                      # S is symmetric; no extra rescaling
gamma = n / m
a, b = sigma2 * (1 - np.sqrt(gamma))**2, sigma2 * (1 + np.sqrt(gamma))**2

print("S shape:", S.shape, f" gamma = {gamma}")
print(f"MP edges: a = {a:.4f}, b = {b:.4f}")
print(f"eigenvalue range: [{eig.min():.4f}, {eig.max():.4f}]")

## Problem 3

Continue with the same financial data as Sheet 3, Problem 3 for SPY, QQQ, TLT, and HYG, but use
the period 2018–2019. Store assets in rows and dates in columns.

In [ ]:
ASSETS = ["SPY", "QQQ", "TLT", "HYG"]

prices = pd.read_csv(MARKET_PATH, index_col=0, parse_dates=True).sort_index()
prices.columns = [c.replace("^", "") for c in prices.columns]
prices = prices[ASSETS]

log_changes = pd.concat(
    {a: np.log(prices[a].dropna()).diff().dropna() for a in ASSETS},
    axis=1, join="inner",
)
log_changes = log_changes.loc["2018-01-01":"2019-12-31"]

R = log_changes.T.to_numpy()    # 4 x m: assets in rows, dates in columns
dates = log_changes.index
n, m = R.shape

print("R shape:", R.shape)
print("first date:", dates[0].date(), " last date:", dates[-1].date())

### a)

Standardize each asset using its sample mean and standard deviation over this period, and compute
the $4 \times 4$ sample correlation matrix $\widehat{\mathbf C}$.

In [ ]:
mean = R.mean(axis=1, keepdims=True)
std = R.std(axis=1, keepdims=True)      # divisor m
Z = (R - mean) / std

C_hat = Z @ Z.T / m

print("max |C_hat - np.corrcoef(R)| =", np.max(np.abs(C_hat - np.corrcoef(R))))
pd.DataFrame(C_hat, index=ASSETS, columns=ASSETS)

### b)

Calculate the unit noise MP upper edge using $\gamma = 4/m$ and add it as a reference line to a
scree plot. Explain why this line should be treated as a heuristic. Mention at least three
assumptions of the iid MP noise model that might not hold for daily financial returns.

In [ ]:
gamma = n / m
_, b_upper = mp_edges(gamma, sigma2=1.0)

lam = np.linalg.eigvalsh(C_hat)[::-1]    # decreasing order
ks = np.arange(1, n + 1)

print(f"gamma = 4/{m} = {gamma:.4f},  MP upper edge b = {b_upper:.4f}")
print("eigenvalues:", np.round(lam, 4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, lam, "o-", label="eigenvalues of C_hat")
ax.axhline(b_upper, color="red", ls="--", label=f"MP upper edge b = {b_upper:.3f}")
ax.set_xticks(ks)
ax.set_xlabel("k")
ax.set_ylabel("lambda_k")
ax.set_title("Scree plot of the correlation matrix (2018-2019)")
ax.legend()
plt.show()

**Discussion.**